# Bài 52: Mô hình thị giác-ngôn ngữ (VLM) (Vision-Language Models (VLMs))

Mọi bộ phân loại từ đầu đến giờ đều dự đoán một trong một tập nhỏ, cố định các lớp được định sẵn lúc huấn luyện (Bài 36). **CLIP** ("Learning Transferable Visual Models From Natural Language Supervision", <a href="../references.html#radford-2021">Radford et al., 2021</a><span class="landmark-paper">&#9733;</span>) phá vỡ ràng buộc đó bằng cách huấn luyện một bộ mã hóa ảnh và một bộ mã hóa văn bản *đồng thời*, với một hàm mất mát tương phản (Bài 49) kéo vector nhúng của mỗi ảnh về phía vector nhúng của chú thích (caption) của nó và đẩy nó ra xa mọi chú thích khác trong lô. Thành quả: phân loại trở thành việc viết tên các lớp dưới dạng lời nhắc văn bản (text prompt) và kiểm tra độ tương đồng &mdash; không cần phần đầu phân loại, không cần huấn luyện lại, hoạt động trên các danh mục mà mô hình chưa bao giờ thấy mẫu có nhãn.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Ảnh được ghép cặp với chú thích (Images paired with captions)
Một "ngôn ngữ" tổng hợp nhỏ xíu (8 từ) tạo ra các chú thích đơn giản như "a photo of a large plus" hay "a photo of a small circle" cho mỗi ảnh. Đây rõ ràng không phải ngôn ngữ tự nhiên thực sự, nhưng nó vận hành chính xác cùng cơ chế mà CLIP dùng trên các cặp ảnh-chú thích thật thu thập từ web: mỗi mẫu huấn luyện là một cặp (ảnh, văn bản), và không gì khác.

In [ ]:
VOCAB = ['a', 'photo', 'of', 'plus', 'circle', 'small', 'large', 'noisy']
WORD_TO_ID = {w: i for i, w in enumerate(VOCAB)}

def make_image(shape_type, cx, cy, size, rng):
    img = np.zeros((size, size), dtype=np.float32)
    if shape_type == 'plus':
        r = size // 5
        img[cy-1:cy+2, cx-r:cx+r+1] = 1.0
        img[cy-r:cy+r+1, cx-1:cx+2] = 1.0
    else:
        r = size // 4
        yy, xx = np.mgrid[0:size, 0:size]
        img[((xx-cx)**2 + (yy-cy)**2) <= r**2] = 1.0
    return np.clip(img + rng.normal(0, 0.1, img.shape), 0, 1).astype(np.float32)

def caption_for(shape_type, size_desc):
    return [WORD_TO_ID[w] for w in ['a', 'photo', 'of', 'a', size_desc, shape_type]]

def make_dataset(rng_local, n, image_size=16):
    imgs, captions = [], []
    for _ in range(n):
        shape_type = rng_local.choice(['plus', 'circle'])
        size_desc = rng_local.choice(['small', 'large'])
        cx, cy = image_size // 2, image_size // 2
        img = make_image(shape_type, cx, cy, image_size, rng_local)
        if size_desc == 'small':
            small = img[4:12, 4:12]
            img = np.zeros_like(img)
            img[4:12, 4:12] = small
        imgs.append(img)
        captions.append(caption_for(shape_type, size_desc))
    return np.array(imgs, dtype=np.float32), captions

rng = np.random.default_rng(7)
X_train, cap_train = make_dataset(rng, 400)
X_test, cap_test = make_dataset(rng, 150)

fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
for ax, im, cap in zip(axes, X_train[:4], cap_train[:4]):
    ax.imshow(im, cmap='gray'); ax.axis('off')
    ax.set_title(' '.join(VOCAB[w] for w in cap), fontsize=7)
plt.show()

## Hai bộ mã hóa, một không gian nhúng chung (Two encoders, one shared embedding space)
Một bộ mã hóa ảnh (theo mẫu CNN ở Bài 34) và một bộ mã hóa văn bản (một bảng tra nhúng từ (word embedding), gộp trung bình trên chú thích — bộ mã hóa văn bản đơn giản nhất có thể; CLIP thực tế dùng Transformer, Bài 46) đều ánh xạ các đầu vào rất khác nhau của chúng vào *cùng* một không gian vector kích thước cố định. Huấn luyện kéo các cặp (ảnh, chú thích) khớp nhau trong một lô lại gần nhau và đẩy mọi cặp không khớp ra xa — chính xác là `nt_xent_loss` ở Bài 49, chỉ là áp dụng qua hai phương thức (modality) khác nhau thay vì hai góc nhìn tăng cường của một ảnh.

In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self, out_dim=16):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(), nn.AdaptiveMaxPool2d(1),
        )
        self.proj = nn.Linear(32, out_dim)

    def forward(self, x):
        return self.proj(self.conv(x).flatten(1))

class TextEncoder(nn.Module):
    def __init__(self, vocab_size, out_dim=16, embed_dim=8):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.proj = nn.Linear(embed_dim, out_dim)

    def forward(self, token_ids):
        return self.proj(self.embed(token_ids).mean(dim=1))  # bag-of-words text encoder

def clip_loss(img_emb, txt_emb, temperature=0.1):
    img_emb = F.normalize(img_emb, dim=1)
    txt_emb = F.normalize(txt_emb, dim=1)
    logits = img_emb @ txt_emb.T / temperature       # (batch, batch) similarity matrix
    targets = torch.arange(img_emb.shape[0])          # the diagonal is the true pairing
    loss_i2t = F.cross_entropy(logits, targets)        # "which caption matches this image?"
    loss_t2i = F.cross_entropy(logits.T, targets)       # "which image matches this caption?"
    return (loss_i2t + loss_t2i) / 2

def train_clip(seed, epochs=300, lr=0.01, batch_size=64):
    torch.manual_seed(seed)
    img_enc = ImageEncoder()
    txt_enc = TextEncoder(len(VOCAB))
    opt = torch.optim.Adam(list(img_enc.parameters()) + list(txt_enc.parameters()), lr=lr)
    caps_tensor = torch.tensor(cap_train)
    n = len(X_train)
    for epoch in range(epochs):
        idx = np.random.default_rng(epoch).permutation(n)[:batch_size]
        imgs = torch.tensor(X_train[idx]).unsqueeze(1)
        caps = caps_tensor[idx]
        loss = clip_loss(img_enc(imgs), txt_enc(caps))
        opt.zero_grad()
        loss.backward()
        opt.step()
    return img_enc, txt_enc, loss.item()

img_enc, txt_enc, final_loss = train_clip(seed=0)
print(f'final CLIP loss: {final_loss:.3f}')

## Phân loại zero-shot (Zero-shot classification)
Không có phần đầu phân loại nào từng được huấn luyện. Để phân loại một ảnh kiểm tra, hãy nhúng một lời nhắc văn bản cho mỗi lớp ứng viên ("a photo of a large plus", "a photo of a large circle") và chọn lời nhắc nào có vector nhúng giống nhất với vector nhúng của ảnh — cùng ý tưởng láng-giềng-gần-nhất-trong-không-gian-nhúng như ví dụ truy xuất chú ý ở Bài 45, chỉ là qua các phương thức khác nhau.

In [ ]:
prompts = {'plus': caption_for('plus', 'large'), 'circle': caption_for('circle', 'large')}
plus_id, circle_id = WORD_TO_ID['plus'], WORD_TO_ID['circle']
true_labels = [c[-1] for c in cap_test]  # last word id encodes the true shape
X_test_t = torch.tensor(X_test).unsqueeze(1)

def zero_shot_eval(img_encoder, txt_encoder):
    with torch.no_grad():
        prompt_embs = {k: F.normalize(txt_encoder(torch.tensor([v])), dim=1) for k, v in prompts.items()}
        img_embs = F.normalize(img_encoder(X_test_t), dim=1)
        sims_plus = (img_embs @ prompt_embs['plus'].T).squeeze(-1)
        sims_circle = (img_embs @ prompt_embs['circle'].T).squeeze(-1)
        preds = torch.where(sims_plus > sims_circle, plus_id, circle_id)
    return sum(p.item() == t for p, t in zip(preds, true_labels)) / len(true_labels)

acc_trained = zero_shot_eval(img_enc, txt_enc)

torch.manual_seed(0)  # same init recipe, but never trained -- the baseline
untrained_img_enc = ImageEncoder()
untrained_txt_enc = TextEncoder(len(VOCAB))
acc_untrained = zero_shot_eval(untrained_img_enc, untrained_txt_enc)

print(f'zero-shot accuracy, trained encoders:   {acc_trained:.1%}')
print(f'zero-shot accuracy, untrained encoders: {acc_untrained:.1%}  (baseline)')

Các bộ mã hóa đã huấn luyện giải hoàn hảo bài toán phân loại zero-shot (không cần mẫu); các bộ chưa huấn luyện chỉ ở mức đoán ngẫu nhiên — mục tiêu tương phản ảnh-văn bản, không phải một bộ phân loại cụ thể nào, đang làm toàn bộ công việc. Đây là cơ chế thực sự đằng sau kết quả nổi bật của CLIP thật: nó có thể phân loại vào *bất kỳ* tập danh mục nào mô tả được bằng lời, kể cả những danh mục chưa bao giờ xuất hiện như một lớp có nhãn trong khi huấn luyện, chỉ bằng cách thay đổi lời nhắc văn bản lúc suy luận. Nó cũng là cách tiêu chuẩn để xây dựng phần đầu thị giác của một **mô hình thị giác-ngôn ngữ (vision-language model — VLM)** — một bộ mã hóa ảnh kiểu CLIP đóng băng đưa vào một mô hình ngôn ngữ là nền tảng cho các hệ thống có thể trả lời câu hỏi về ảnh, và cùng mục tiêu tương phản ảnh-văn bản ghép cặp đó, áp dụng cho các đề xuất vùng thay vì toàn bộ ảnh, là một con đường dẫn tới phát hiện và phân đoạn từ vựng mở (open-vocabulary) (Bài 55).

### Bài tập (Exercises)
1. Thêm một danh mục hình dạng thứ ba (ví dụ "square", theo mẫu ở Bài 36) vào `make_dataset`, huấn luyện lại, và thêm một lời nhắc `'square'` vào phần đánh giá zero-shot. Phân loại zero-shot 3 lớp có còn hoạt động tốt như trường hợp 2 lớp không?
2. Thử một lời nhắc dùng một tổ hợp từ *chưa bao giờ thấy cùng nhau* trong huấn luyện — ví dụ nếu các chú thích huấn luyện chỉ luôn ghép `'noisy'` với không hình nào, hãy dựng các biến thể `caption_for` dùng `'noisy'` và kiểm tra xem thứ hạng độ tương đồng của mô hình có còn hợp lý không. Điều này kiểm tra gì về việc mô hình đã học nghĩa từ có tính kết hợp (compositional) hay chỉ ghi nhớ chú thích?
3. Bộ mã hóa văn bản của bài này gộp trung bình các nhúng từ, hoàn toàn bỏ qua thứ tự từ (`'a large plus'` và `'plus large a'` tạo ra cùng một vector nhúng). Điều đó có tạo ra khác biệt với các chú thích cụ thể này không? Hãy phác thảo một thay đổi (gợi ý: Bài 46) làm cho bộ mã hóa văn bản nhạy với thứ tự, và mô tả một cặp chú thích mà ở đó sự nhạy với thứ tự thực sự quan trọng.